# arXiv Parser — Colab (Milestone 2)

Notebook này parse các LaTeX source đã crawl ở Milestone 1 thành ba bảng:
, , , kèm , ,
 và .

## Cải tiến & Đồng bộ mới nhất

- **Hỗ trợ Multi-range Queue**: Tự động phát hiện các thư mục dải đã cào trong  (hoặc ).
- **Chống nghẽn Drive FUSE (I/O Stall)**: Shards trung gian  được ghi tại ổ đĩa cục bộ Colab (), sau khi streaming gộp thành 3 file Parquet lớn mới đồng bộ sang Google Drive.
- **Bảo toàn dữ liệu ()**: Giữ lại các version bị lỗi include nhẹ để không làm hụt dữ liệu trích dẫn cho Milestone 3.
- **Cấu hình **: Mặc định  để chạy toàn bộ dải (có thể đổi thành  để smoke test).


## 1. Mount Drive và chọn thư mục dự án

Sửa `PROJECT_DIR` nếu thư mục dự án trên Drive của bạn khác.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_DIR = '/content/drive/MyDrive/arxiv'

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    print('Không phải Colab: bỏ qua mount Drive')

root = Path(PROJECT_DIR).expanduser()
root.mkdir(parents=True, exist_ok=True)
os.chdir(root)
print('Project root:', root)


## 2. Lấy/cập nhật code và cài dependency

Cell này fetch `main` và `reset --hard` trong `code/`, nên sửa đổi local trong
`code/` sẽ bị ghi đè. Nếu vừa pull code mới, restart runtime trước khi import lại.


In [ ]:
repo_url = 'https://github.com/NewB-S2deeC0de/arXiv.git'
branch = 'main'
repo = Path('code')
if not (repo / '.git').exists():
    if repo.exists():
        import shutil
        shutil.rmtree(repo)
    subprocess.run(['git', 'clone', '-b', branch, repo_url, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'remote', 'set-url', 'origin', repo_url], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', branch], check=True)
    subprocess.run(['git', '-C', str(repo), 'reset', '--hard', f'origin/{branch}'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'src/requirements.txt')], check=True)
if str(repo / 'src') not in sys.path:
    sys.path.insert(0, str(repo / 'src'))
print('Code commit:', subprocess.check_output(['git', '-C', str(repo), 'log', '-1', '--oneline'], text=True).strip())


## 3. Tự động phát hiện dữ liệu Milestone 1 trong 

Tự động quét toàn bộ các thư mục range đã cào xong trong  (hoặc ).
Chỉ đọc discovery và marker/checksum hiện có, không gọi API và không sửa dữ liệu.


In [ ]:
from pathlib import Path
from parser.adapter import list_publications, publications_with_records
from parser import run_parser

# Tự động quét tìm các target folder trong datasets/ (từ M1 multi-range crawler)
dataset_dirs = sorted([p for p in Path('datasets').glob('*') if (p / 'data').is_dir()])
if not dataset_dirs and Path('runtime/data').is_dir():
    dataset_dirs = [Path('runtime')]

print(f'Tìm thấy {len(dataset_dirs)} thư mục dữ liệu M1:')
for idx, d in enumerate(dataset_dirs, start=1):
    pubs = list_publications(d)
    print(f'  [{idx}] {d.name} -> {len(pubs)} publications')

if not dataset_dirs:
    raise FileNotFoundError('Chưa tìm thấy thư mục dữ liệu M1 nào trong datasets/ hoặc runtime/. Hãy kiểm tra kết quả của crawl_colab.ipynb.')

# Xem thử 5 bài đầu tiên của range đầu tiên
sample_dir = dataset_dirs[0]
print(f'Sample publications từ {sample_dir.name}:')
for arxiv_id, records in publications_with_records(sample_dir, limit=5):
    print(arxiv_id, '| versions:', [r.version for r in records],
          '| status:', [str(r.status) for r in records])


## 4. Chạy parser M2

- : Mặc định chạy toàn bộ dữ liệu cào. Nếu muốn smoke test nhanh, đặt .
- : Giữ checkpoint và bỏ qua các bài đã parse thành công.
- : Ghi các file shard  vào ổ đĩa cục bộ Colab () để **tránh hoàn toàn hiện tượng treo I/O Drive FUSE**, sau đó tự động copy kết quả Parquet hoàn chỉnh sang Google Drive.
- : Mặc định giữ lại các version  để tối đa hóa số bài trích xuất BibTeX cho Milestone 3.


In [ ]:
import shutil
import tempfile

# ==============================================================================
# CẤU HÌNH RUN PARSER
# ==============================================================================
LIMIT = None           # Đặt None để chạy toàn bộ; đổi thành số nguyên (ví dụ 10) nếu muốn test nhanh
RESUME = True          # True: tiếp tục từ checkpoint; False: xoá shard và parse lại từ đầu
WORKERS = 2           # Chạy song song 2 nhân CPU (Google Colab Free có sẵn 2 vCPUs để tăng gấp đôi tốc độ)

# [CẢNH BÁO KHI TẮT INCLUDE_PARTIAL]:
# Nếu đặt INCLUDE_PARTIAL = False, toàn bộ các bài LaTeX có cấu trúc phức tạp
# bị phân loại 'partial' (như thiếu file style phụ, dùng macro include động...)
# sẽ bị loại bỏ hoàn toàn khỏi bảng elements và bibtex. Điều này sẽ làm hụt
# đáng kể số lượng bài báo và trích dẫn chuẩn bị cho Milestone 3.
INCLUDE_PARTIAL = True # MẶC ĐỊNH True để giữ tối đa dữ liệu

# Chống nghẽn Drive FUSE: Ghi các file shard tạm trên đĩa máy ảo Colab thay vì ghi trực tiếp Drive FUSE
USE_LOCAL_SCRATCH = True

if LIMIT is not None and LIMIT <= 0:
    raise ValueError('LIMIT phải > 0 hoặc None')

def log(message):
    print(message, flush=True)

manifests = {}
for target_dir in dataset_dirs:
    target_name = target_dir.name
    print(f'=======================================================')
    print(f'BẮT ĐẦU PARSE: {target_name}')
    print(f'=======================================================')

    drive_out = target_dir / 'parsed'
    drive_out.mkdir(parents=True, exist_ok=True)

    if USE_LOCAL_SCRATCH:
        local_scratch = Path(f'/content/scratch_parsed/{target_name}')
        local_scratch.mkdir(parents=True, exist_ok=True)
        if RESUME and (drive_out / 'checkpoint.json').exists():
            shutil.copyfile(drive_out / 'checkpoint.json', local_scratch / 'checkpoint.json')
            if (drive_out / '_parts').exists() and not (local_scratch / '_parts').exists():
                shutil.copytree(drive_out / '_parts', local_scratch / '_parts')
        out_target = local_scratch
    else:
        out_target = drive_out

    manifest = run_parser(
        target_dir,
        output_root=out_target,
        limit=LIMIT,
        resume=RESUME,
        include_partial=INCLUDE_PARTIAL,
        workers=WORKERS,
        log=log
    )
    manifests[target_name] = manifest

    if USE_LOCAL_SCRATCH:
        print(f'[Hoàn tất {target_name}] Đồng bộ các file kết quả Parquet sang Google Drive: {drive_out} ...')
        for file in out_target.glob('*.parquet'):
            shutil.copyfile(file, drive_out / file.name)
        for file in out_target.glob('*.csv'):
            shutil.copyfile(file, drive_out / file.name)
        for file in out_target.glob('*.json'):
            shutil.copyfile(file, drive_out / file.name)
        if (out_target / 'checkpoint.json').exists():
            shutil.copyfile(out_target / 'checkpoint.json', drive_out / 'checkpoint.json')
        # Tránh nghẽn I/O Drive: Không copy trực tiếp hàng nghìn folder con của _parts sang Drive.
        # Toàn bộ dữ liệu bảng M2 đã nằm trọn vẹn trong các file .parquet và .json ở trên.
        # Nếu bạn muốn lưu bản sao lưu của _parts để resume sau này, có thể nén thành 1 file tar:
        BACKUP_PARTS_TAR = False
        if BACKUP_PARTS_TAR and (out_target / '_parts').exists():
            print(f'Nén và sao lưu _parts về {drive_out / "_parts.tar"} ...')
            subprocess.run(['tar', '-cf', str(drive_out / '_parts.tar'), '-C', str(out_target), '_parts'], check=True)
        print(f'[Đã lưu xong vào Drive]: {drive_out}')

print('=== TOÀN BỘ CÁC RANGE ĐÃ ĐƯỢC PARSE XONG ===')


## 5. Thống kê từ manifest

Bao gồm coverage version, số element/BibTeX trước và sau dedup, thời gian chạy,
peak RSS và dung lượng output.


In [ ]:
import json

for name, manifest in manifests.items():
    print(f'--- Thống kê cho {name} ---')
    print(json.dumps({
        'papers_total': manifest['papers_total'],
        'papers_processed': manifest['papers_processed'],
        'versions': manifest['versions'],
        'elements': manifest['elements'],
        'bibtex': manifest['bibtex'],
        'timing': manifest['timing'],
        'memory': manifest['memory'],
        'storage': manifest['storage'],
        'validation': manifest['validation'],
    }, indent=2, ensure_ascii=False))
    print('Errors:', len(manifest['errors']))
    for error in manifest['errors'][:5]:
        print(' ', error)


## 6. Xem ba bảng và coverage

Đọc trực tiếp Parquet bằng pandas; không sửa dữ liệu.


In [ ]:
import pandas as pd

try:
    display
except NameError:
    display = print

first_dir = dataset_dirs[0]
first_out = first_dir / 'parsed'
print(f'=== Đọc dữ liệu mẫu từ {first_out} ===')
elements = pd.read_parquet(first_out / 'elements.parquet')
hierarchy = pd.read_parquet(first_out / 'hierarchy.parquet')
bibtex = pd.read_parquet(first_out / 'bibtex.parquet')
coverage = pd.read_csv(first_out / 'coverage.csv')

print('elements:', len(elements), '| hierarchy:', len(hierarchy), '| bibtex:', len(bibtex))
display(elements.head(10))
print('Element kinds:')
display(elements['kind'].value_counts())
print('Coverage theo publication:')
display(coverage.head(10))


## 7. Hiệu năng và dung lượng

`peak_rss_bytes` là peak RSS của process (Colab Linux dùng `resource`), không
phải đo riêng từng publication. `parts_bytes` là kích thước shard;
`final_tables_bytes` là output cuối cùng.


In [ ]:
import json
import os

def directory_bytes(path):
    return sum(os.path.getsize(p) for p in Path(path).rglob('*') if p.is_file())

for target_dir in dataset_dirs:
    out_dir = target_dir / 'parsed'
    manifest = manifests.get(target_dir.name)
    if not manifest or not out_dir.exists():
        continue
    hierarchy_file = out_dir / 'hierarchy.parquet'
    if hierarchy_file.exists():
        h_df = pd.read_parquet(hierarchy_file)
        shared = h_df.groupby('element_id')['version'].nunique()
        print(f'[{target_dir.name}] Elements xuất hiện ở >= 2 version: {int((shared >= 2).sum())}')

    versions = manifest['versions']
    stats = {
        'elapsed_seconds': manifest['timing']['elapsed_seconds'],
        'average_seconds_per_publication': manifest['timing']['average_seconds_per_publication'],
        'peak_rss_bytes': manifest['memory']['peak_rss_bytes'],
        'final_tables_bytes': directory_bytes(out_dir),
        'parts_bytes': manifest['storage']['parts_bytes'],
        'parse_success_rate_versions': (
            versions['versions_parsed'] / versions['versions_total']
            if versions['versions_total'] else None),
    }
    print(f'Thống kê hiệu năng [{target_dir.name}]:')
    print(json.dumps(stats, indent=2, ensure_ascii=False))


## Ghi chú vận hành

### Output
- Bảng cuối: `runtime/parsed/{elements,hierarchy,bibtex}.parquet`
- Thống kê: `coverage.csv`, `errors.json`, `manifest.json`, `checkpoint.json`
- Validation: `manifest['validation']` (schema, unique id, hierarchy FK, citation refs)
- Shard tạm: `runtime/parsed/_parts/`
- Không ghi vào `runtime/data`, `runtime/progress`, `runtime/exports`.

### Resume
- Publication chỉ bị bỏ qua khi có trong checkpoint và đủ ba shard.
- `RESUME=False` xoá checkpoint + `_parts/` rồi parse lại; dùng khi đổi logic parser.
- Version `no_tex` được ghi nhận là empty, không bị bỏ khỏi coverage.
- Version `failed`/`missing` được liệt kê trong `errors.json`, không dừng run.

### Khi nào cần chạy lại
- Sau khi crawler thêm version mới hoặc sửa source: chạy `RESUME=True` để parse phần mới.
- Sau khi đổi code parser/cleanup/dedup: chạy `RESUME=False` để tránh trộn output cũ.
- CLI tương đương: `PYTHONPATH=src python -m parser.cli --persistent-root runtime --limit 10`.
